# Thesis: Entity-Aware A-RAG with Evidence Verification

**Đề tài**: Nghiên cứu cải tiến mô hình A-RAG dựa trên theo dõi thực thể và kiểm chứng bằng chứng trong hỏi đáp đa bước

**Branch**: `thesis-entity-evidence-arag`

**Thứ tự chạy**: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11

## Cell 1: Clone Repo / Pull Code

In [6]:
import os, subprocess

REPO_URL = "https://github.com/trangdx2602/arag.git"
DATA_URL = "https://huggingface.co/datasets/Ayanami0730/rag_test"
REPO_DIR = "/content/arag"
BRANCH = "thesis-entity-evidence-arag"
FORCE_RECLONE = False  # Set True nếu muốn xóa và clone lại từ đầu

# --- Clone / pull repo ---
if FORCE_RECLONE and os.path.exists(REPO_DIR):
    import shutil; shutil.rmtree(REPO_DIR)
    print("Removed existing repo for re-clone.")

if os.path.exists(REPO_DIR):
    remote = subprocess.run(["git", "-C", REPO_DIR, "remote", "get-url", "origin"],
                            capture_output=True, text=True).stdout.strip()
    if "trangdx2602" not in remote:
        import shutil; shutil.rmtree(REPO_DIR)
        print(f"Wrong remote ({remote}), re-cloning from {REPO_URL}...")
        !git clone {REPO_URL} {REPO_DIR}
        !cd {REPO_DIR} && git checkout {BRANCH}
    else:
        print("Repo already exists — pulling latest...")
        !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone {REPO_URL} {REPO_DIR}
    !cd {REPO_DIR} && git checkout {BRANCH}

# --- Download dataset from HuggingFace ---
DATA_DIR = f"{REPO_DIR}/data"
if not os.path.exists(f"{DATA_DIR}/musique/chunks.json"):
    print("Downloading dataset from HuggingFace (2-5 phút)...")
    !pip install huggingface_hub -q
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="Ayanami0730/rag_test",
        repo_type="dataset",
        local_dir=DATA_DIR,
        ignore_patterns=["*.git*"],
    )
    print("Dataset downloaded.")
else:
    print("Dataset already present.")

!ls {REPO_DIR}

# Change working directory so relative paths in YAML configs resolve correctly
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Cloning into '/content/arag'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 210 (delta 17), reused 17 (delta 13), pack-reused 163 (from 1)
Receiving objects: 100% (210/210), 2.67 MiB | 26.25 MiB/s, done.
Resolving deltas: 100% (77/77), done.
Branch 'thesis-entity-evidence-arag' set up to track remote branch 'thesis-entity-evidence-arag' from 'origin'.
Switched to a new branch 'thesis-entity-evidence-arag'


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Dataset downloaded.
assets	      configs  docs	  pyproject.toml  scripts  tests
CITATION.cff  data     notebooks  README.md	  src
Working directory: /content/arag


## Cell 2: Cài Môi Trường + Set API Key

In [7]:
!pip install -e "/content/arag[full]" -q

import os

# ============================================================
# NHẬP API KEY CỦA BẠN TẠI ĐÂY
# (Nếu dùng Colab Secrets thì để trống — xem hướng dẫn bên dưới)
# ============================================================
ARAG_API_KEY_MANUAL = ""   # <-- Dán API key vào đây, VD: "sk-proj-abc123..."
ARAG_MODEL_MANUAL    = "llama-3.3-70b-versatile"
ARAG_BASE_URL_MANUAL = "https://api.groq.com/openai/v1"

if ARAG_API_KEY_MANUAL:
    os.environ["ARAG_API_KEY"]  = ARAG_API_KEY_MANUAL
    os.environ["ARAG_MODEL"]    = ARAG_MODEL_MANUAL or "gpt-4o-mini"
    os.environ["ARAG_BASE_URL"] = ARAG_BASE_URL_MANUAL or "https://api.openai.com/v1"
    print("API key set manually.")
else:
    try:
        from google.colab import userdata
        key = userdata.get("ARAG_API_KEY")
        if not key:
            raise ValueError("Empty")
        os.environ["ARAG_API_KEY"]  = key
        os.environ["ARAG_MODEL"]    = userdata.get("ARAG_MODEL") or "gpt-4o-mini"
        os.environ["ARAG_BASE_URL"] = userdata.get("ARAG_BASE_URL") or "https://api.openai.com/v1"
        print("API key loaded from Colab Secrets.")
    except Exception:
        print("WARNING: ARAG_API_KEY chưa được set!")
        print("  → Cách 1: Điền ARAG_API_KEY_MANUAL ở trên rồi chạy lại cell này")
        print("  → Cách 2: Sidebar trái → biểu tượng 🔑 Secrets → Add new secret")

print("API key set  :", bool(os.environ.get("ARAG_API_KEY")))
print("Model        :", os.environ.get("ARAG_MODEL", "gpt-4o-mini"))
print("Base URL     :", os.environ.get("ARAG_BASE_URL", "https://api.openai.com/v1"))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for arag (pyproject.toml) ... done
API key set manually.
API key set  : True
Model        : llama-3.3-70b-versatile
Base URL     : https://api.groq.com/openai/v1


## Cell 3: Mount Google Drive

In [8]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_RESULTS_DIR = "/content/drive/MyDrive/thesis_arag_results"
import os
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {DRIVE_RESULTS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results will be saved to: /content/drive/MyDrive/thesis_arag_results


## Cell 4: Build Index HotpotQA / MuSiQue (GPU)

In [ ]:
import os, gc, subprocess, sys
REPO_DIR = "/content/arag"

# Auto-detect GPU — fall back to CPU if not available
import torch
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8 if DEVICE.startswith("cuda") else 32
print(f"Device: {DEVICE}  |  batch_size: {BATCH_SIZE}")
if DEVICE.startswith("cuda"):
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def build_index(dataset):
    ret = subprocess.run([
        sys.executable, f"{REPO_DIR}/scripts/build_index.py",
        "--chunks",    f"{REPO_DIR}/data/{dataset}/chunks.json",
        "--output",    f"{REPO_DIR}/data/{dataset}/index",
        "--model",     "Qwen/Qwen3-Embedding-0.6B",
        "--device",    DEVICE,
        "--batch-size", str(BATCH_SIZE),
    ])
    if ret.returncode != 0:
        print(f"ERROR: {dataset} index build FAILED (returncode={ret.returncode})")
    else:
        print(f"OK: {dataset} index built.")
    return ret.returncode == 0

print("\nBuilding MuSiQue index...")
ok1 = build_index("musique")

if DEVICE.startswith("cuda"):
    torch.cuda.empty_cache(); gc.collect()
    print("VRAM cleared.")

print("\nBuilding HotpotQA index...")
ok2 = build_index("hotpotqa")

if ok1 and ok2:
    print("\nBoth indexes built successfully.")
else:
    print("\nWARNING: one or more indexes failed — check errors above.")

## Cell 5: Chạy Naive RAG

In [10]:
## Cell 4b: Xóa kết quả cũ (chạy cell này trước khi chạy lại variants)
# Bỏ qua nếu đây là lần chạy đầu tiên

import shutil, os
REPO_DIR = "/content/arag"

VARIANTS = [
    "naive_rag", "arag_baseline", "arag_entity_tracker",
    "arag_evidence_checker", "arag_entity_evidence_full",
]
DATASET = "musique"

for v in VARIANTS:
    d = f"{REPO_DIR}/results/thesis/{v}_{DATASET}"
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Cleared: {d}")
    else:
        print(f"Not found (OK): {d}")

print("Done — ready to re-run Cells 5–9.")

Not found (OK): /content/arag/results/thesis/naive_rag_musique
Not found (OK): /content/arag/results/thesis/arag_baseline_musique
Not found (OK): /content/arag/results/thesis/arag_entity_tracker_musique
Not found (OK): /content/arag/results/thesis/arag_evidence_checker_musique
Not found (OK): /content/arag/results/thesis/arag_entity_evidence_full_musique
Done — ready to re-run Cells 5–9.


In [11]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 1

# Pull latest code (get CWD fix if not done via Cell 1)
!git -C /content/arag pull origin thesis-entity-evidence-arag -q

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_base.yaml --variant naive_rag --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/naive_rag_musique --limit {LIMIT} --workers {WORKERS}

print("Naive RAG done.")

Variant: naive_rag
Total: 20 | Completed: 0 | Pending: 20
Running with 1 workers...
naive_rag: 100% 20/20 [00:02<00:00,  9.33it/s]
Results saved to: /content/arag/results/thesis/naive_rag_musique/predictions.jsonl
Naive RAG done.


## Cell 6: Chạy A-RAG Baseline

In [12]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_base.yaml --variant arag_baseline --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/arag_baseline_musique --limit {LIMIT} --workers {WORKERS}

print("A-RAG Baseline done.")

Variant: arag_baseline
Total: 20 | Completed: 0 | Pending: 20
Running with 5 workers...
arag_baseline: 100% 20/20 [00:00<00:00, 21.39it/s]
Results saved to: /content/arag/results/thesis/arag_baseline_musique/predictions.jsonl
A-RAG Baseline done.


## Cell 7: Chạy A-RAG + Entity Tracker

In [13]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_entity.yaml --variant arag_entity_tracker --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/arag_entity_tracker_musique --limit {LIMIT} --workers {WORKERS}

print("A-RAG + Entity Tracker done.")

Variant: arag_entity_tracker
Total: 20 | Completed: 0 | Pending: 20
Running with 5 workers...
arag_entity_tracker: 100% 20/20 [00:00<00:00, 21.49it/s]
Results saved to: /content/arag/results/thesis/arag_entity_tracker_musique/predictions.jsonl
A-RAG + Entity Tracker done.


## Cell 8: Chạy A-RAG + Evidence Checker

In [14]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_evidence.yaml --variant arag_evidence_checker --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/arag_evidence_checker_musique --limit {LIMIT} --workers {WORKERS}

print("A-RAG + Evidence Checker done.")

Variant: arag_evidence_checker
Total: 20 | Completed: 0 | Pending: 20
Running with 5 workers...
arag_evidence_checker: 100% 20/20 [00:00<00:00, 20.05it/s]
Results saved to: /content/arag/results/thesis/arag_evidence_checker_musique/predictions.jsonl
A-RAG + Evidence Checker done.


## Cell 9: Chạy A-RAG + Full (ET + EV)

In [15]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_full.yaml --variant arag_entity_evidence_full --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/arag_entity_evidence_full_musique --limit {LIMIT} --workers {WORKERS}

print("A-RAG Full done.")

Variant: arag_entity_evidence_full
Total: 20 | Completed: 0 | Pending: 20
Running with 5 workers...
arag_entity_evidence_full: 100% 20/20 [00:00<00:00, 21.44it/s]
Results saved to: /content/arag/results/thesis/arag_entity_evidence_full_musique/predictions.jsonl
A-RAG Full done.


## Cell 10: Evaluate và Xuất Bảng So Sánh

In [16]:
import subprocess, json
REPO_DIR = "/content/arag"

# Quick contain-match comparison (no LLM needed)
!python {REPO_DIR}/scripts/thesis/compare_results.py \
    --results {REPO_DIR}/results/thesis/ \
    --dataset musique

# Optional: LLM-based accuracy evaluation (costs money)
# for variant in ["naive_rag", "arag_baseline", "arag_entity_tracker",
#                 "arag_evidence_checker", "arag_entity_evidence_full"]:
#     !python {REPO_DIR}/scripts/eval.py \
#         --predictions {REPO_DIR}/results/thesis/{variant}_musique/predictions.jsonl \
#         --config {REPO_DIR}/configs/thesis/musique_base.yaml \
#         --workers 5

# Display comparison JSON
import json
cmp_file = f"{REPO_DIR}/results/thesis/comparison_musique.json"
try:
    with open(cmp_file) as f:
        cmp = json.load(f)
    import pandas as pd
    rows = []
    for v, stats in cmp.items():
        if stats:
            rows.append({"variant": v, **stats})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
except Exception as e:
    print(f"Cannot display table: {e}")


=== Thesis Results Comparison — MUSIQUE ===

  naive_rag: 20 predictions loaded from /content/arag/results/thesis/naive_rag_musique/predictions.jsonl
  arag_baseline: 20 predictions loaded from /content/arag/results/thesis/arag_baseline_musique/predictions.jsonl
  arag_entity_tracker: 20 predictions loaded from /content/arag/results/thesis/arag_entity_tracker_musique/predictions.jsonl
  arag_evidence_checker: 20 predictions loaded from /content/arag/results/thesis/arag_evidence_checker_musique/predictions.jsonl
  arag_entity_evidence_full: 20 predictions loaded from /content/arag/results/thesis/arag_entity_evidence_full_musique/predictions.jsonl

---------------------------------------------------------------------------------------------------------------------------
Variant                       N       Contain-Acc   Avg Loops    Avg Tokens    Avg Cost($)   Avg Entities    Avg Coverage  
------------------------------------------------------------------------------------------------

## Cell 11: Copy Results về Google Drive

In [17]:
import shutil, os
REPO_DIR = "/content/arag"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/thesis_arag_results"

src = f"{REPO_DIR}/results/thesis"
dst = f"{DRIVE_RESULTS_DIR}/results_thesis"

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Results copied to: {dst}")
else:
    print("No results to copy yet.")

# List saved files
for root, dirs, files in os.walk(dst):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"  {path.replace(dst, '')} ({size:,} bytes)")

Results copied to: /content/drive/MyDrive/thesis_arag_results/results_thesis
  /comparison_musique.json (1,294 bytes)
  /naive_rag_musique/predictions.jsonl (15,358 bytes)
  /arag_baseline_musique/predictions.jsonl (9,128 bytes)
  /arag_entity_tracker_musique/predictions.jsonl (9,128 bytes)
  /arag_evidence_checker_musique/predictions.jsonl (9,128 bytes)
  /arag_entity_evidence_full_musique/predictions.jsonl (9,128 bytes)
